# Analisis Eksperimen — G09 Sepakat

Notebook ini membaca hasil run di `output/**/*.json` (ditulis oleh `src/experiments.py` atau `src/main.py`) dan
menampilkan:

1. tabel hasil per bagian EXPERIMENTS §4,
2. plot per algoritma (objective vs iterasi, `e^(ΔE/T)` untuk SA, max/mean populasi untuk GA),
3. state awal dan akhir run terbaik tiap algoritma,
4. ringkasan antar-algoritma.

Tidak ada search yang dijalankan di sini; semua dibuat dari JSON. Jalankan eksperimen dulu dari root repo:

```bash
python src/experiments.py
```

Variabel lingkungan opsional: `EXPERIMENTS_OUTPUT` (folder hasil, default `output/`) dan `EXPERIMENTS_QUICK=1`
(membaca hasil `python src/experiments.py --quick`).

In [ ]:
import io
import os
import sys
from itertools import accumulate
from pathlib import Path

from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure

# root repo: notebook bisa dibuka dari folder notebooks/ atau dari root
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").is_file())
sys.path.insert(0, str(ROOT / "src"))

from config import OUTPUT_DIR
from core.problem import load_problem
from experiments import comparison_groups, load_result, machine_info, planned_runs, summary_sections
from visualization.plots import plot_objective, plot_run, plot_summary
from visualization.state_view import draw_state

OUT = Path(os.environ.get("EXPERIMENTS_OUTPUT", OUTPUT_DIR))
QUICK = os.environ.get("EXPERIMENTS_QUICK") == "1"


def show(fig):
    """Tampilkan Figure matplotlib (modul visualisasi tidak memakai pyplot)."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=110)
    display(Image(data=buffer.getvalue()))


print("output :", OUT)
print("mesin  :", machine_info())

## 1. Memuat hasil

Semua `output/**/*.json` dibaca, lalu dicocokkan dengan rencana run di `experiments.planned_runs()` supaya tiap
hasil tahu grupnya (bagian EXPERIMENTS). Run yang belum ada (misalnya Bonus 1 selama objective ETA belum
tersedia) ditampilkan di bawah.

In [ ]:
files = sorted(OUT.rglob("*.json"))
results = {path.relative_to(OUT).with_suffix("").as_posix(): load_result(path) for path in files}

plan = planned_runs(quick=QUICK)
pairs = [(run, results[f"{run.algo}/{run.run_id}"]) for run in plan if f"{run.algo}/{run.run_id}" in results]
planned = {f"{run.algo}/{run.run_id}" for run in plan}
missing = [f"{run.algo}/{run.run_id}" for run in plan if f"{run.algo}/{run.run_id}" not in results]
extra = sorted(set(results) - planned)

print(f"{len(files)} file JSON, {len(pairs)} dari {len(plan)} run yang direncanakan")
print("belum ada :", ", ".join(missing) or "-")
print("di luar rencana (mis. run manual lewat CLI):", ", ".join(extra) or "-")

## 2. Tabel per bagian EXPERIMENTS §4

Kolomnya sama dengan tabel di `docs/EXPERIMENTS.md`, jadi bisa langsung disalin. Isi yang sama juga ditulis
ke `output/summary.md` oleh runner.

In [ ]:
display(Markdown("\n".join(summary_sections(pairs))))

## 3. Plot per algoritma

Satu baris per algoritma/konfigurasi, satu kolom per seed. GA: maksimum dan rata-rata populasi per generasi.
Random Restart: objective terbaik sejauh ini. SA: plot `e^(ΔE/T)` ada di gambar per run (`*_plot.png`) dan di
bagian 3b.

In [ ]:
def label(run):
    if run.algo == "ga":
        return f"ga pop {run.params['population']} iter {run.params['iterations']}"
    return f"{run.algo} ({run.instance.stem})"


by_label = {}
for run, result in pairs:
    by_label.setdefault(label(run), []).append(result)

for name, group in by_label.items():
    group = sorted(group, key=lambda r: r["seed"])
    fig = Figure(figsize=(5 * len(group), 3.4), layout="constrained")
    for ax, result in zip(fig.subplots(1, len(group), squeeze=False)[0], group):
        plot_objective(ax, result)
        ax.set_title(f"seed {result['seed']}: {ax.get_title()}", fontsize=8)
    fig.suptitle(name)
    show(fig)

### 3b. Simulated Annealing: objective dan `e^(ΔE/T)`

In [ ]:
for run, result in pairs:
    if run.algo == "sa":
        show(plot_run(result))

## 4. State awal dan akhir run terbaik tiap algoritma

Run terbaik = objective akhir tertinggi di antara seed. Panel per geladak (multi-kapal: satu panel per kapal),
ditambah daftar kendaraan di luar kapal.

In [ ]:
problems = {}
best = {}
for run, result in pairs:
    if run.objective != "fee":
        continue
    name = label(run)
    if name not in best or result["final_objective"] > best[name][1]["final_objective"]:
        best[name] = (run, result)

for name, (run, result) in best.items():
    problem = problems.setdefault(run.instance, load_problem(run.instance))
    for key in ("initial_state", "final_state"):
        show(draw_state(problem, result[key], title=f"{key} | {name} | seed {result['seed']}"))

## 5. Ringkasan antar-algoritma

Box plot objective akhir (titik = tiap run) dan rata-rata durasi. GA diwakili konfigurasi dengan rata-rata
objective terbaik. Lalu perbandingan konfigurasi GA, dan kurva konvergensi terhadap jumlah evaluasi objective
(lebih adil daripada iterasi, EXPERIMENTS §1).

In [ ]:
compare = comparison_groups(pairs)
if compare:
    owner = {id(r): name for name, group in compare.items() for r in group}
    show(plot_summary([r for group in compare.values() for r in group], group=lambda r: owner[id(r)]))

for group, key, name in (("ga_iter", "iterations", "iter"), ("ga_pop", "population", "pop")):
    results_ga = sorted((res for run, res in pairs if group in run.groups), key=lambda r: r["params"][key])
    if results_ga:
        show(plot_summary(results_ga, group=lambda r: f"{name} {r['params'][key]}"))

In [ ]:
# konvergensi: x = evaluasi objective. Evaluasi dianggap tersebar rata antar-iterasi (pendekatan).
# Sumbu y dibatasi ke 0..UB supaya bagian akhir pencarian terlihat (objective awal bisa sangat negatif).
fig = Figure(figsize=(8, 4.5), layout="constrained")
ax = fig.subplots()
upper = None
for name, group in compare.items():
    result = max(group, key=lambda r: r["final_objective"])
    curve = result["history"].get("best", result["history"]["objective"])
    best_so_far = list(accumulate(curve, max))
    xs = [1 + result["evaluations"] * k / max(1, len(curve) - 1) for k in range(len(curve))]
    ax.plot(xs, best_so_far, linewidth=1, label=f"{name} (seed {result['seed']})")
    upper = result["upper_bound"]
if upper is not None:
    ax.axhline(upper, color="gray", linestyle=":", linewidth=0.8, label="upper bound")
    ax.set_ylim(0, upper * 1.05)
ax.set_xscale("log")
ax.set_xlabel("evaluasi objective + 1 (skala log)")
ax.set_ylabel("objective")
ax.legend(fontsize=7)
ax.grid(alpha=0.3)
show(fig)

## 6. Analisis

Isi bagian ini Rabu 21 setelah run resmi selesai, mengikuti checklist EXPERIMENTS §5:

- perbandingan objective akhir terhadap upper bound (gap) antar-algoritma,
- kecepatan konvergensi (iterasi dan evaluasi) dan durasi,
- konsistensi hasil antar-seed,
- pengaruh populasi dan banyak iterasi pada GA,
- frekuensi stuck dan peluang `e^(ΔE/T)` pada SA,
- varian HC (Bonus 2) dan multi-kapal (Bonus 3).